# 12. EDA và trực quan hóa đồ thị (task 2.1, 2.2, 2.3, GĐ C)

Cho **cả hai bộ**: NF-ToN-IoT-v3 (chính) và NF-BoT-IoT-v3. Mọi hàm vẽ nằm ở `scripts/eda.py`, notebook chỉ gọi.

- **Mức flow** (đọc CSV gốc theo khối bằng pyarrow, hai lượt): phân bố nhãn, số flow theo thời gian, boxplot log(1 + x) theo lớp, heatmap tương quan, mutual information, bảng min/median/max/% thiếu.
- **Mức đồ thị** (đồ thị đã dựng ở GĐ B và notebook 04): số node và cạnh mỗi cửa sổ, bậc vào/ra theo lớp, edge homophily và class-insensitive homophily.
- **Cửa sổ mẫu**: bình thường, DDoS, dò quét (và backdoor với NF-ToN-IoT-v3), ≤ 300 node, cùng layout seed; thêm 1 sơ đồ quy trình flow → cửa sổ → đồ thị.

Kaggle CPU (không bật GPU), bật Internet, gắn `athena21/netflow-v3-datasets` và output của `gnn-iot-ids-04-build-graphs`, `gnn-iot-ids-11-build-graphs-bot`.
Output: `gd2_<bộ>_<tên>.png`, `eda_summary.json` (số liệu và nhận xét từng hình), `run_info.json`.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager

SMOKE = False      # True: chỉ đọc vài khối đầu của CSV để chạy thử nhanh (không dùng cho kết quả chính thức)
WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"          # ngoài /kaggle/working để không lẫn vào output
INFO = {"steps_s": {}, "device": "CPU", "gpu": None, "smoke": SMOKE}
T_START = time.time()


def peak_ram_gb():
    """RAM đỉnh (GB) của tiến trình notebook và mọi tiến trình con đã kết thúc (ru_maxrss tính bằng KB trên Linux)."""
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()                # ghi sau mỗi bước, lỗi giữa chừng vẫn còn run_info.json
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-7000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

try:
    import torch
    if torch.cuda.is_available():
        INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
except ImportError:
    pass
INFO["cpu_count"] = os.cpu_count()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"], "| CPU:", INFO["cpu_count"], "| SMOKE:", SMOKE)

## 1. Kiểm thử các hàm tính toán của `eda.py`
(homophily tính tay trên đồ thị nhỏ, chọn cửa sổ mẫu, định dạng số)

In [ ]:
with step("pytest_eda"):
    r = run([sys.executable, "-m", "pytest", "-q", "tests/test_eda.py"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## 2. Tìm đầu vào và chạy EDA cho từng bộ
Mỗi bộ chạy trong một tiến trình riêng để giải phóng RAM giữa hai bộ.

In [ ]:
DATASETS = ["nf_ton_iot_v3", "nf_bot_iot_v3"]
CSV_NAME = {"nf_ton_iot_v3": "NF-ToN-IoT-v3.csv", "nf_bot_iot_v3": "NF-BoT-IoT-v3.csv"}
INPUTS = {}
for ds in DATASETS:
    csv = next(glob.iglob(f"/kaggle/input/**/{CSV_NAME[ds]}", recursive=True))
    gdir = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
    INPUTS[ds] = (csv, gdir)
    print(ds, "\n  CSV:", csv, f"({os.path.getsize(csv) / 2**30:.2f} GB)", "\n  đồ thị:", gdir)

for ds in DATASETS:
    csv, gdir = INPUTS[ds]
    with step(f"eda_{ds}"):
        run([sys.executable, "-I", "scripts/eda.py", "run", ds, csv, gdir, WORK] + (["--smoke"] if SMOKE else []), cwd=REPO)

## 3. Gộp, vẽ sơ đồ quy trình và kiểm tra tiêu chí

In [ ]:
with step("gop_va_kiem_tra"):
    run([sys.executable, "-I", "scripts/eda.py", "merge", WORK, *DATASETS], cwd=REPO)

    from PIL import Image
    summ = json.load(open(f"{WORK}/eda_summary.json", encoding="utf-8"))
    figs = summ["figures"]
    rows, problems = [], []
    for f in figs:
        path = f"{WORK}/{f['file']}"
        with Image.open(path) as im:
            dpi = round(im.info.get("dpi", (0, 0))[0])
            size = f"{im.width}x{im.height}"
        mb = os.path.getsize(path) / 2**20
        rows.append((f["id"], f["file"], size, dpi, round(mb, 2)))
        if dpi < 150:
            problems.append(f"{f['file']}: dpi {dpi} < 150")
        if mb >= 5:
            problems.append(f"{f['file']}: {mb:.1f} MB ≥ 5 MB")
        if not f["comment"].strip():
            problems.append(f"{f['id']}: thiếu nhận xét")
        if f["axis_label_problems"]:
            problems.append(f"{f['id']}: {f['axis_label_problems']}")
    print(f"{'id':38s} {'file':52s} {'kích thước':>10s} {'dpi':>4s} {'MB':>6s}")
    for r_ in rows:
        print(f"{r_[0]:38s} {r_[1]:52s} {r_[2]:>10s} {r_[3]:4d} {r_[4]:6.2f}")
    print("\nHomophily:", json.dumps({k: {m: round(v[m], 4) for m in ("edge_homophily", "class_insensitive_homophily", "adjusted_homophily")}
                                      for k, v in summ["homophily"].items()}, ensure_ascii=False))
    for k, v in summ["homophily"].items():
        print(f"  {k}: {v['meaning_for_gnn']}")
    if problems:
        print("\nVẤN ĐỀ:", *problems, sep="\n  ")
    INFO["n_figures"] = len(figs)
    INFO["figures_per_dataset"] = summ["n_figures_per_dataset"]
    INFO["homophily"] = {k: {m: v[m] for m in ("edge_homophily", "class_insensitive_homophily")} for k, v in summ["homophily"].items()}
    INFO["problems"] = problems
    assert len(figs) >= 12, f"chỉ có {len(figs)} hình"
    assert not problems, problems

## 4. Tổng kết và run_info.json

In [ ]:
save_info()
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())
run(["bash", "-c", f"ls -la {WORK} | head -60"])